<h2>Impact (v0.64) - 'Impact (v0.64).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 'v0.61' made use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less. A new 'composite' weight field has been added in Trade (v0.61) which only introduces the VtW-derived values where 'netWgt' is observed to be zero, however this resulted in only a tiny increase in recorded weight of around 1% for Angola, whereas the full VtW-derived field was ~50% greater. 

'v0.62' makes use of a new 'transport cost' field derived using ad-valorem transport cost rates identified from UNCTAD's Trade-and-Transport database. With these, we can now introduce another results disaggregation that presents NZF costs relative to expected transport costs, which is then fully in-line with how impacts are expressed in UNCTAD's CIA Task 3 work. 'v0.63' includes an assessment of 'Cost Intensity' matching process (similar to CIF-FOB matching), plus metrics on 'fac_vessel_types' and 'average sea-distance' (introduced in Trade v0.63). 'v0.64' has been modified throughout to account for a new scenario, namely the 'Japan' proposal, plus to remove the 'Low-then-High' scenario. In terms of the Impact notebook, this remains the same as 'v0.63'.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

version = "v0.64"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
voys_fem_dir = "/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments (v0.2)/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_v0.2.csv"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"
res_mlc = False

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "132"
iso_country = "Cabo Verde"
year = 2023
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 2.500*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."
os.makedirs(output_dir+"{0}/results/".format(iso_country)) if not os.path.exists(output_dir+"{0}/results/".format(iso_country)) else "Output results folder already exists."

## 1.2 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv(voys_fem_dir, dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,0,5034525,413781699,CNTAX,156,CHN,CN,China,Taixing,CNNKG,156,CHN,CN,China,Nanjing,2018-01-01 06:00:00,2018-01-01 19:00:00,13.0,1.0,14.0,62.568523,0.0,0.0,0.0,3.635661,0.338483,3.974144,0.0,0.0,0.0,0.0,0.0,0.0,0.155243,0.014453,11.655930,1.085176,12.741107,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,0.169696,Refrigerated bulk,3786.0,dwt,3881.0,NaN,NaN,NaN,NaN,2,2000-5999,1959.0,Panama,SSD,4,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,242828.439338,7661.320668,7934.549945,7805.304846,7691.950531,7580.696290,7471.542086,7360.199865,7255.307537,7148.350725,7667.458038,12670.391275,12727.642802,14093.647297,14570.027623,14398.931835,14189.590704,14139.488873,14161.462401,7665.715352,8412.516422,8946.6

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {
    "reporterCode":"str", "partnerCode":"str", "partner2Code":"str", 
    "cmdCode":"str", "HS2":"str", "HS4":"str", "hs2":"str", "hs4":"str", "hs6":"str",
    "isic":"str", "sitc":"str", "gtap":"str", "cpc":"str"
}
if res_mlc == False:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_{1}.csv".format(iso_country, version), dtype=trade_im_d)
else:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_mlc_{1}.csv".format(iso_country, version), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Cabo Verde': 

Number of trade records present in dataset: 14219
Number of partner countries present in dataset: 107
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1617.3m
Total weight within the dataset: 2396.6 million kilograms

Distance to market: 526.3 nm (min) to 13059.4 nm (max), 3690.3 nm (observed mean)
Average distance to market (volume-weighted): 3626.5 nm
Average distance to market (value-weighted): 4005.9 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm
0,0,132,Cabo Verde,620,Portugal,000,World,Import,Sea,"291211 | Aldehydes; acyclic,",291211,kg,1283.0,1283.0,3113.898,2860.660,3113.898,"Portugal | 291211 | Aldehydes; acyclic,",29,29 | Organic chemicals,Portugal | 29 | Organic chemicals,2912,"2912 | Aldehydes, whether o","Portugal | 2912 | Aldehydes, whether o",29,2912,291211,Methanal (formaldehyde),"Petroleum, Chemical and Non-Metallic Mineral P...",2411,"Manufacture of basic chemicals, except fertili...",51621,Acyclic aldehydes without other oxygen function,33,"CRP - Chemical, rubber, plastic products",3417,"Ethers, alcohol peroxides, ether peroxides, ep...",2.1758,0.4596,both,0.412024,1431.147521,1283.0,[Container],['Container'],2575.0,1600.030325,2289.879433
1,1,132,Cabo Verde,724,Spain,000,World,Import,Sea,"291211 | Aldehydes; acyclic,",291211,kg,10.0,10.0,20.987,15.682,20.987,"Spain | 291211 | Aldehydes; acyclic,",29,29 | Organic chemicals,Spain | 29 | Organic chemicals,2912,"2912 | Aldehydes, whether o","Spain | 2912 | Aldehydes, whether o",29,2912,291211,Methanal (formaldehyde),"Petroleum, Chemical and Non-Metallic Mineral P...",2411,"Manufacture of basic chemicals, except fertili...",51621,Acyclic aldehydes without other oxygen function,33,"CRP - Chemical, rubber, plastic products",3417,"Ethers, alcohol peroxides, ether peroxides, ep...",2.1758,0.4596,both,0.476485,9.645625,10.0,[Container],['Container'],2711.0,1684.536781,16.248410


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 328
Total size of the fleet: 242 vessels
8 vessel types contained in the dataset: ['General cargo' 'Refrigerated bulk' 'Container' 'Oil tanker'
 'Chemical tanker' 'Bulk carrier' 'Vehicle' 'Liquefied gas tanker']
Total transport work capacity: 33918.4 million deadweight-tonne nautical miles
Total cargo transported: 7292889.9 tonnes
Total transport work: 18794.2 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 2268.3 EJ
Total CO2 produced: 174408.2 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,3958,7310507,450511000,ESCAR,724,ESP,ES,Spain,Cartagena,CVGRA,132,CPV,CV,Cabo Verde,Porto Grande,2018-09-18 00:00:00,2018-09-24 01:00:00,145.0,31.0,176.0,1807.722020,64.765333,0.000394,64.765727,5.5057,2.3533,7.8590,0.0,0.0,0.0,0.0,0.0,0.0,2.838660,0.100502,219.330521,7.545907,226.876428,150,Europe,039,Southern Europe,high_income,0,0,002,Africa,202,Sub-Saharan Africa,low_mid_income,0,1,2.939162,General cargo,5395.0,dwt,5579.0,NaN,NaN,NaN,NaN,2,5000-9999,1973.0,Lebanon,MSD,1,19.1,0.639731,0.013245,0.013561,0.013411,0.01328,0.013152,0.013025,0.012897,0.012775,0.012652,0.013243,0.018992,0.018721,0.020055,0.022205,0.021986,0.021305,0.020103,0.019359,0.013243,0.014075,0.014424,0.016556,0.018232,0.021456,0.021072,0.020684,0.019424,0.013243,0.013542,0.014012,0.015662,0.017612,0.02233,0.021597,0.021774,0.021549,0.013244,0.013551,0.013846,0.014985,0.015408,0.016447,0.01858,0.018382,0.01842,0.013244,0.01391,0.014148,0.014473,0.01505,0.015474,0.015636,0.015745,0.015819,1.008528e+07,133579.010189,136765.615585,135258.261999,133936.238697,132638.708031,131365.669575,130067.112816,128843.779572,127596.368754,133563.640073,191538.717427,188807.531683,202257.498984,223942.512098,221732.537530,214863.

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_arr_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_arr_voys_vess = int_arr_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_arr_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

                if int_arr_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_arr_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the importer.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "direct35_usd_dwtnm_"+year))
    
    if cost_int_match_desc_bool == False:
        trade_im["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True
    
    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd
    trade_im["direct_usd_"+year] = direct_usd

print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print("\nNumber of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.direct_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_agg = trade_im.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg["vol_kg_vtw_pct"] = round(100 * trade_im_agg.vol_kg_vtw / trade_im_agg.vol_kg_vtw.sum(), 1)
trade_im_agg["cifvalue_pct"] = round(100 * trade_im_agg.cifvalue / trade_im_agg.cifvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_im_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5)
))
trade_im_agg_2 = trade_im.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg_2["vol_kg_vtw_pct"] = round(100 * trade_im_agg_2.vol_kg_vtw / trade_im_agg_2.vol_kg_vtw.sum(), 1)
trade_im_agg_2["cifvalue_pct"] = round(100 * trade_im_agg_2.cifvalue / trade_im_agg_2.cifvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_im_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

[]

Global voyages table used for the following vessel types found: 

['Cruise' 'Other liquids tankers' 'Ro-Ro']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['8462 | Machine-tools (inclu' '2935 | Sulphonamides'
 '3002 | Human blood; animal ' '3006 | Pharmaceutical goods'
 '8487 | Machinery parts; not' '5516 | Woven fabrics of art'
 '3105 | Fertilizers; mineral' '8514 | Industrial or labora'
 '3215 | Ink; printing, writi' '8517 | Telephone sets, incl'
 '3307 | Perfumery, cosmetic ' '8525 | Transmission apparat'
 '3402 | Organic surface-acti' '3507 | Enzymes; prepared en'
 '8528 | Monitors and project' '8539 | Lamps; electric fila'

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,cifvalue_pct
0,['Oil tanker'],1. Primary Int. Inv. with Vess Type,47.1,45.1
1,['Container'],1. Primary Int. Inv. with Vess Type,40.2,37.9
2,"['Container', 'Bulk carrier']",1. Primary Int. Inv. with Vess Type,3.8,1.8
3,['Chemical tanker'],1. Primary Int. Inv. with Vess Type,3.3,3.3
4,['all'],3. Primary Int. Inv. without Vess Type,2.2,5.2


Rows with NA: $84.674 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,132,Cabo Verde,620,Portugal,000,World,Import,Sea,"291211 | Aldehydes; acyclic,",291211,kg,1283.0,1283.0,3113.898,2860.660,3113.898,"Portugal | 291211 | Aldehydes; acyclic,",29,29 | Organic chemicals,Portugal | 29 | Organic chemicals,2912,"2912 | Aldehydes, whether o","Portugal | 2912 | Aldehydes, whether o",29,2912,291211,Methanal (formaldehyde),"Petroleum, Chemical and Non-Metallic Mineral P...",2411,"Manufacture of basic chemicals, except fertili...",51621,Acyclic aldehydes without other oxygen function,33,"CRP - Chemical, rubber, plastic products",3417,"Ethers, alcohol peroxides, ether peroxides, ep...",2.1758,0.4596,both,0.412024,1431.147521,1283.0,[Container],['Container'],2575.0,1600.030325,2289.879433,1. Primary Int. Inv. with Vess Type,11.968809,11.947911,11.949953,11.949953,11.959318,11.959318,12.519982,22.189337,13.809942,12.821246,12.856066,13.178057,12.259262,21.731058,14.535835,13.314607,13.009064,13.598245,12.030597,23.631052,18.025133,15.764145,14.584818,14.173398,11.806168,25.125600,20.675483,19.374682,15.614925,15.190336,11.585976,25.461276,24.325996,24.535153,16.928201,15.956958,11.361370,24.937946,23.898193,26.183982,19.368111,16.249425,11.149776,22.245253,23.185475,24.295958,18.213018,16.441420,10.934016,20.434166,20.771864,25.910220,21.405477,16.550296
1,1,132,Cabo Verde,724,Spain,000,World,Import,Sea,"291211 | Aldehydes; acyclic,",291211,kg,10.0,10.0,20.987,15.682,20.987,"Spain | 291211 | Aldehydes; acyclic,",29,29 | Organic chemicals,Spain | 29 | Organic chemicals,2912,"2912 | Aldehydes, whether o","Spain | 2912 | Aldehydes, whether o",29,2912,291211,Methanal (formaldehyde),"Petroleum, Chemical and Non-Metallic Mineral P...",2411,"Manufacture of basic chemicals, except fertili...",51621,Acyclic aldehydes without other oxygen function,33,"CRP - Chemical, rubber, plastic products",3417,"Ethers, alcohol peroxides, ether peroxides, ep...",2.1758,0.4596,both,0.476485,9.645625,10.0,[Container],['Container'],2711.0,1684.536781,16.248410,1. Primary Int. Inv. with Vess Type,0.084928,0.084779,0.084794,0.084794,0.084860,0.084860,0.088839,0.157450,0.097992,0.090976,0.091223,0.093508,0.086989,0.154198,0.103143,0.094477,0.092309,0.096490,0.085366,0.167680,0.127902,0.111858,0.103490,0.100571,0.083774,0.178285,0.146708,0.137478,0.110800,0.107787,0.082211,0.180667,0.172611,0.174095,0.120118,0.113227,0.080617,0.176953,0.169576,0.185795,0.137431,0.115302,0.079116,0.157847,0.164518,0.172398,0.129235,0.116664,0.077585,0.144996,0.147392,0.183852,0.151888,0.117437


## 2.4 Create Results Tables

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,27468534.91,26997322.04,26521707.16,26068133.66,25597921.02,25137147.61,24703800.67,24259595.02
1,Levy,46421550.71,45759220.64,50288800.42,54528838.58,55868009.80,53927715.04,47323485.66,43617255.12
2,'as-is',29607120.81,31623061.31,37790112.69,44116275.22,51782142.71,50894104.32,48219374.96,44386502.37
3,Brazil,27859923.93,28969107.27,34057735.75,41137132.20,53944271.26,56010558.00,52388485.51,53304459.30
4,LISCR,27979503.65,28439370.04,31703532.90,33467211.18,36075138.10,42759673.63,40635950.50,43389179.68
5,Direct,28709036.98,29596551.46,30761802.18,32800769.68,34469890.02,35055889.13,35403691.77,35623665.20



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.70,1.67,1.64,1.61,1.58,1.55,1.53,1.50
1,Levy,2.87,2.83,3.11,3.37,3.45,3.33,2.93,2.70
2,'as-is',1.83,1.96,2.34,2.73,3.20,3.15,2.98,2.74
3,Brazil,1.72,1.79,2.11,2.54,3.34,3.46,3.24,3.30
4,LISCR,1.73,1.76,1.96,2.07,2.23,2.64,2.51,2.68
5,Direct,1.78,1.83,1.90,2.03,2.13,2.17,2.19,2.20



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.099,1.080,1.061,1.043,1.024,1.005,0.988,0.970
1,Levy,1.857,1.830,2.012,2.181,2.235,2.157,1.893,1.745
2,'as-is',1.184,1.265,1.512,1.765,2.071,2.036,1.929,1.775
3,Brazil,1.114,1.159,1.362,1.645,2.158,2.240,2.096,2.132
4,LISCR,1.119,1.138,1.268,1.339,1.443,1.710,1.625,1.736
5,Direct,1.148,1.184,1.230,1.312,1.379,1.402,1.416,1.425


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)
    
print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,18953015.79,18761898.60,23767093.26,28460704.93,30270088.78,28790567.43,22619684.99,19357660.10
1,'as-is',2138585.89,4625739.28,11268405.54,18048141.56,26184221.69,25756956.71,23515574.30,20126907.35
2,Brazil,391389.02,1971785.24,7536028.59,15068998.54,28346350.24,30873410.39,27684684.85,29044864.28
3,LISCR,510968.73,1442048.01,5181825.74,7399077.52,10477217.08,17622526.02,15932149.83,19129584.66
4,Direct,1240502.07,2599229.42,4240095.02,6732636.02,8871969.00,9918741.53,10699891.10,11364070.18



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,69.00,69.50,89.61,109.18,118.25,114.53,91.56,79.79
1,'as-is',4.52,9.63,15.99,25.83,34.66,39.46,43.31,46.84
2,Brazil,7.79,17.13,42.49,69.23,102.29,102.47,95.19,82.96
3,LISCR,1.42,7.30,28.41,57.81,110.74,122.82,112.07,119.73
4,Direct,1.86,5.34,19.54,28.38,40.93,70.11,64.49,78.85



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.17,1.16,1.47,1.76,1.87,1.78,1.40,1.20
1,'as-is',0.08,0.16,0.26,0.42,0.55,0.61,0.66,0.70
2,Brazil,0.13,0.29,0.70,1.12,1.62,1.59,1.45,1.24
3,LISCR,0.02,0.12,0.47,0.93,1.75,1.91,1.71,1.80
4,Direct,0.03,0.09,0.32,0.46,0.65,1.09,0.99,1.18



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.758,0.750,0.951,1.138,1.211,1.152,0.905,0.774
1,'as-is',0.050,0.104,0.170,0.269,0.355,0.397,0.428,0.455
2,Brazil,0.086,0.185,0.451,0.722,1.047,1.030,0.941,0.805
3,LISCR,0.016,0.079,0.301,0.603,1.134,1.235,1.107,1.162
4,Direct,0.020,0.058,0.207,0.296,0.419,0.705,0.637,0.765


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {
    "reporterCode":"str", "partnerCode":"str", "partner2Code":"str", 
    "cmdCode":"str", "HS2":"str", "HS4":"str", "hs2":"str", "hs4":"str", "hs6":"str",
    "isic":"str", "sitc":"str", "gtap":"str", "cpc":"str"
}
if res_mlc == False:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
else:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_mlc_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Cabo Verde': 

Number of trade records present in dataset: 1651
Number of partner countries present in dataset: 46
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$164.2m
Total weight within the dataset: 150.6 million kilograms

Distance to market: 526.3 nm (min) to 13046.3 nm (max), 2149.3 nm (observed mean)
Average distance to market (volume-weighted): 3385.8 nm
Average distance to market (value-weighted): 2787.3 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm
0,0,132,Cabo Verde,251,France,000,World,Export,Sea,846291 | Machine-tools; press,846291,NaN,0.0,40.0,1779.614,1779.614,1779.614,France | 846291 | Machine-tools; press,84,84 | Machinery and mechan,France | 84 | Machinery and mechan,8462,8462 | Machine-tools (inclu,France | 8462 | Machine-tools (inclu,84,8462,846291,Hydraulic presses for working metal,Electrical and Machinery,2922,Manufacture of machine-tools,73318,"Presses for working metal/metal carbides, n.e.s.",41,OME - Machinery and equipment n.e.c.,44217,"Machine-tools for working metal by forging, ha...",7.4095,0.1350,both,0.022477,240.247890,40.0,[all],['all'],3783.0,2350.646493,564.73786
1,1,132,Cabo Verde,620,Portugal,000,World,Export,Sea,846320 | Machine-tools; for w,846320,NaN,0.0,25.0,438.681,438.681,438.681,Portugal | 846320 | Machine-tools; for w,84,84 | Machinery and mechan,Portugal | 84 | Machinery and mechan,8463,8463 | Machine-tools; n.e.c,Portugal | 8463 | Machine-tools; n.e.c,84,8463,846320,Thread rolling machines for working metal/cerm...,Electrical and Machinery,2922,Manufacture of machine-tools,73393,Thread-rolling machines,41,OME - Machinery and equipment n.e.c.,44218,"Machine-tools n.e.c. for working metal, sinter...",15.2679,0.0655,both,0.056989,28.733605,25.0,"[Container, General cargo]","['Container', 'General cargo']",2575.0,1600.030325,45.97464


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 336
Total size of the fleet: 249 vessels
8 vessel types contained in the dataset: ['General cargo' 'Refrigerated bulk' 'Container' 'Oil tanker'
 'Chemical tanker' 'Bulk carrier' 'Vehicle' 'Liquefied gas tanker']
Total transport work capacity: 39068.0 million deadweight-tonne nautical miles
Total cargo transported: 7488878.2 tonnes
Total transport work: 21481.4 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 2562.9 EJ
Total CO2 produced: 197084.3 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,3959,7310507,450511000,CVGRA,132,CPV,CV,Cabo Verde,Porto Grande,BRVLC,076,BRA,BR,Brazil,Vila Do Conde,2018-09-25 08:00:00,2018-10-05 15:00:00,247.0,45.0,292.0,1907.229781,58.059491,0.000003,58.059494,11.7769,3.8520,15.6289,0.0,0.0,0.0,0.0,0.0,0.0,2.836865,0.164481,218.553997,12.349521,230.903518,002,Africa,202,Sub-Saharan Africa,low_mid_income,0,1,019,Americas,419,Latin America and the Caribbean,upp_mid_income,0,0,3.001346,General cargo,5395.0,dwt,5579.0,NaN,NaN,NaN,NaN,2,5000-9999,1973.0,Lebanon,MSD,1,19.1,0.639731,0.013245,0.013561,0.013411,0.01328,0.013152,0.013025,0.012897,0.012775,0.012652,0.013243,0.018992,0.018721,0.020055,0.022205,0.021986,0.021305,0.020103,0.019359,0.013243,0.014075,0.014424,0.016556,0.018232,0.021456,0.021072,0.020684,0.019424,0.013243,0.013542,0.014012,0.015662,0.017612,0.02233,0.021597,0.021774,0.021549,0.013244,0.013551,0.013846,0.014985,0.015408,0.016447,0.01858,0.018382,0.01842,0.013244,0.01391,0.014148,0.014473,0.01505,0.015474,0.015636,0.015745,0.015819,1.064043e+07,140931.992627,144294.007719,142703.680434,141308.885115,139939.930652,138596.816594,137226.779556,135936.106781,134620.031059,140915.776449,202082.146544,199200.620108,213390.953528,236269

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_dep_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_dep_voys_vess = int_dep_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_dep_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
                
                if int_dep_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_dep_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
    
        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the exporter.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "direct35_usd_dwtnm_"+year))

    if cost_int_match_desc_bool == False:
        trade_ex["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd
    trade_ex["direct_usd_"+year] = direct_usd
    
print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.direct_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_agg = trade_ex.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg["vol_kg_vtw_pct"] = round(100 * trade_ex_agg.vol_kg_vtw / trade_ex_agg.vol_kg_vtw.sum(), 1)
trade_ex_agg["fobvalue_pct"] = round(100 * trade_ex_agg.fobvalue / trade_ex_agg.fobvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_ex_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5)
))
trade_ex_agg_2 = trade_ex.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg_2["vol_kg_vtw_pct"] = round(100 * trade_ex_agg_2.vol_kg_vtw / trade_ex_agg_2.vol_kg_vtw.sum(), 1)
trade_ex_agg_2["fobvalue_pct"] = round(100 * trade_ex_agg_2.fobvalue / trade_ex_agg_2.fobvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_ex_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
trade_ex = trade_ex[(~trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_ex_na.primaryValue.sum(), round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

[]

Global voyages table used for the following vessel types found: 

['Ro-Ro']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['8462 | Machine-tools (inclu' '8528 | Monitors and project'
 '8539 | Lamps; electric fila' '8541 | Semiconductor device'
 '6201 | Overcoats, car-coats' '8716 | Trailers and semi-tr' '8803 | nan'
 '8903 | Yachts and other ves' '9405 | Luminaires and light'
 '9406 | Buildings; prefabric' '6815 | Stone or other miner'
 '9701 | Paintings, drawings ' '7326 | Iron or steel; artic'
 '7419 | Copper; articles the' '7508 | Nickel; articles the'
 '7616 | Aluminium; articles ' '3006 | Pharmaceutical goods'
 '8487 | Mac

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,fobvalue_pct
0,['Oil tanker'],1. Primary Int. Inv. with Vess Type,71.4,42.3
1,['Container'],1. Primary Int. Inv. with Vess Type,27.0,53.1
2,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,0.9,1.8
3,['all'],3. Primary Int. Inv. without Vess Type,0.4,1.7
4,"['Chemical tanker', 'Container']",1. Primary Int. Inv. with Vess Type,0.1,0.2


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,132,Cabo Verde,251,France,000,World,Export,Sea,846291 | Machine-tools; press,846291,NaN,0.0,40.0,1779.614,1779.614,1779.614,France | 846291 | Machine-tools; press,84,84 | Machinery and mechan,France | 84 | Machinery and mechan,8462,8462 | Machine-tools (inclu,France | 8462 | Machine-tools (inclu,84,8462,846291,Hydraulic presses for working metal,Electrical and Machinery,2922,Manufacture of machine-tools,73318,"Presses for working metal/metal carbides, n.e.s.",41,OME - Machinery and equipment n.e.c.,44217,"Machine-tools for working metal by forging, ha...",7.4095,0.1350,both,0.022477,240.247890,40.0,[all],['all'],3783.0,2350.646493,564.73786,3. Primary Int. Inv. without Vess Type,1.995597,1.992940,1.993199,1.993199,1.994390,1.994632,2.063447,3.271753,2.181066,2.068096,2.072546,2.138473,2.031089,3.184332,2.279821,2.153285,2.119339,2.189733,2.001946,3.502136,2.706456,2.492366,2.349493,2.260493,1.973545,3.827139,3.094965,2.938014,2.457768,2.386989,1.945271,3.853812,3.658730,3.802225,2.664122,2.482243,1.916635,3.688163,3.560367,3.788282,3.052352,2.516670,1.889687,3.401526,3.478310,3.745263,2.990989,2.539569,1.861988,3.187207,3.236183,3.709531,3.084949,2.552981
1,1,132,Cabo Verde,620,Portugal,000,World,Export,Sea,846320 | Machine-tools; for w,846320,NaN,0.0,25.0,438.681,438.681,438.681,Portugal | 846320 | Machine-tools; for w,84,84 | Machinery and mechan,Portugal | 84 | Machinery and mechan,8463,8463 | Machine-tools; n.e.c,Portugal | 8463 | Machine-tools; n.e.c,84,8463,846320,Thread rolling machines for working metal/cerm...,Electrical and Machinery,2922,Manufacture of machine-tools,73393,Thread-rolling machines,41,OME - Machinery and equipment n.e.c.,44218,"Machine-tools n.e.c. for working metal, sinter...",15.2679,0.0655,both,0.056989,28.733605,25.0,"[Container, General cargo]","['Container', 'General cargo']",2575.0,1600.030325,45.97464,1. Primary Int. Inv. with Vess Type,0.298134,0.297784,0.297814,0.297814,0.297973,0.297973,0.308966,0.500153,0.330808,0.312639,0.312529,0.321344,0.303842,0.484022,0.343873,0.324293,0.318398,0.329457,0.299348,0.528487,0.414135,0.374912,0.352578,0.340580,0.294938,0.569121,0.467435,0.443902,0.369200,0.360368,0.290611,0.574081,0.547829,0.563054,0.395616,0.375094,0.286197,0.556434,0.534312,0.575150,0.453209,0.380647,0.282038,0.508475,0.529025,0.555615,0.438495,0.384291,0.277798,0.477030,0.482575,0.569612,0.475801,0.386469


## 3.4 Create Results Tables

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)

    
print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1413651.53,1390709.26,1365699.31,1342131.80,1317149.36,1293135.32,1270548.56,1247517.05
1,Levy,2383550.97,2360022.44,2599251.39,2822733.06,2918933.41,2813211.89,2434576.77,2241178.81
2,'as-is',1520983.80,1637786.45,1940323.47,2279158.75,2672140.43,2630912.71,2467308.64,2280933.80
3,Brazil,1434486.59,1491987.91,1755293.04,2117036.95,2804688.90,2913807.92,2703866.28,2730244.41
4,LISCR,1441240.26,1466116.62,1633512.57,1720668.41,1851749.75,2225856.13,2107726.88,2207728.90
5,Direct,1477413.56,1525257.27,1585712.67,1690871.56,1780139.20,1810713.40,1827026.96,1836095.43



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.86,0.85,0.83,0.82,0.80,0.79,0.77,0.76
1,Levy,1.45,1.44,1.58,1.72,1.78,1.71,1.48,1.37
2,'as-is',0.93,1.00,1.18,1.39,1.63,1.60,1.50,1.39
3,Brazil,0.87,0.91,1.07,1.29,1.71,1.77,1.65,1.66
4,LISCR,0.88,0.89,0.99,1.05,1.13,1.36,1.28,1.34
5,Direct,0.90,0.93,0.97,1.03,1.08,1.10,1.11,1.12



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.057,0.056,0.055,0.054,0.053,0.052,0.051,0.050
1,Levy,0.095,0.094,0.104,0.113,0.117,0.113,0.097,0.090
2,'as-is',0.061,0.066,0.078,0.091,0.107,0.105,0.099,0.091
3,Brazil,0.057,0.060,0.070,0.085,0.112,0.117,0.108,0.109
4,LISCR,0.058,0.059,0.065,0.069,0.074,0.089,0.084,0.088
5,Direct,0.059,0.061,0.063,0.068,0.071,0.072,0.073,0.073


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:

    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)


print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,969899.45,969313.18,1233552.08,1480601.25,1601784.06,1520076.57,1164028.21,993661.75
1,'as-is',107332.27,247077.19,574624.16,937026.95,1354991.08,1337777.38,1196760.08,1033416.74
2,Brazil,20835.06,101278.64,389593.73,774905.15,1487539.54,1620672.60,1433317.72,1482727.35
3,LISCR,27588.73,75407.35,267813.26,378536.61,534600.39,932720.81,837178.32,960211.85
4,Direct,63762.03,134548.01,220013.36,348739.76,462989.85,517578.08,556478.40,588578.38



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,68.61,69.70,90.32,110.32,121.61,117.55,91.62,79.65
1,'as-is',7.59,17.77,42.08,69.82,102.87,103.45,94.19,82.84
2,Brazil,1.47,7.28,28.53,57.74,112.94,125.33,112.81,118.85
3,LISCR,1.95,5.42,19.61,28.20,40.59,72.13,65.89,76.97
4,Direct,4.51,9.67,16.11,25.98,35.15,40.03,43.80,47.18



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.59,0.59,0.75,0.90,0.98,0.93,0.71,0.61
1,'as-is',0.07,0.15,0.35,0.57,0.83,0.81,0.73,0.63
2,Brazil,0.01,0.06,0.24,0.47,0.91,0.99,0.87,0.90
3,LISCR,0.02,0.05,0.16,0.23,0.33,0.57,0.51,0.58
4,Direct,0.04,0.08,0.13,0.21,0.28,0.32,0.34,0.36



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.039,0.039,0.049,0.059,0.064,0.061,0.047,0.040
1,'as-is',0.004,0.010,0.023,0.037,0.054,0.054,0.048,0.041
2,Brazil,0.001,0.004,0.016,0.031,0.060,0.065,0.057,0.059
3,LISCR,0.001,0.003,0.011,0.015,0.021,0.037,0.033,0.038
4,Direct,0.003,0.005,0.009,0.014,0.019,0.021,0.022,0.024
